# Now lets use different models with openapi wrappers

Now we get to more detail:

1. Different models

2. Structured Outputs

3. Guardrails

In [1]:
from dotenv import load_dotenv
import os
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
from typing import Dict
import sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content
from pydantic import BaseModel

In [2]:
load_dotenv(override=True)

True

In [3]:
openapi_api_key = os.getenv("OPENAPI_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")

In [5]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AI


In [6]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [7]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OLLAMA_BASE_URL = "https://api.openai.com/v1/ollama/"

In [13]:
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
ollama_client = AsyncOpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

gemini_model = OpenAIChatCompletionsModel(model=gemini_client, openai_client=gemini_client)
ollama_model = OpenAIChatCompletionsModel(model=ollama_client, openai_client=ollama_client)

In [15]:
# create the three agents
sale_agent1 = Agent(model=gemini_model, instructions=instructions1, name="Gemini Sales Agent")
sale_agent2 = Agent(model='gpt-4o-mini', instructions=instructions2, name="GPT-4o Mini Sales Agent")
sale_agent3 = Agent(model=ollama_model, instructions=instructions3, name="Ollama Sales Agent")

In [18]:
description = "Write a cold sales email"
# convert these egmail generator agents to tools
tool1 = sale_agent1.as_tool(tool_name="gemini_email_generator", tool_description=description)
tool2 = sale_agent2.as_tool(tool_name="gpt4o_mini_email_generator", tool_description=description)
tool3 = sale_agent3.as_tool(tool_name="ollama_email_generator", tool_description=description)

In [20]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("tangotew@gmail.com")  # Change to your verified sender
    to_email = To("tangogatdet76@gmail.com")  # Change to your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [22]:
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")

In [ ]:
email_tools = [subject_tool, html_tool, send_html_email]

In [26]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."

# create handoff agent that can use the tools to format and branch off to take its own actions to send the email

email_agent = Agent(
  name="Email Manager Agent",
  instructions=instructions,
  model="gpt-4o-mini",
  tools=email_tools,
  handoff_description="Convert an email to HTML and send it"
)

In [27]:
tools = [tool1, tool2, tool3]
handoffs = [email_agent]

In [30]:
sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""

sale_manager = Agent(
  name="Sales Manager",
  instructions=sales_manager_instructions,
  model="gpt-4o-mini",
  tools=tools,
  handoffs=handoffs
)

message = "Send out a cold sales email addressed to Dear CEO from Tom"

with trace("Automated different models"):
    response = await Runner.run(sale_manager, message)

# Guardrails
We can use guardrails to ensure that the output of our agents is in a specific format, or to ensure that they don't do certain things. This is especially important when we have agents that are using tools that can have real-world consequences, like sending emails.

In [31]:
# we can code our own guardrails or make them another agent
class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str
  
guardrail_agent = Agent(
  name= "Name Check",
  instructions="Check if the user is including someone's personal name in what they want you to do.",
  output_type=NameCheckOutput, # specify the output type so that the agent will be forced to respond in this format and we can easily parse it
  model="gpt-4o-mini"
)

In [33]:
@input_guardrail
async def guardrail_against_name(ctx, agent, message):
  result = await Runner.run(guardrail_agent, message, context=ctx.context)
  is_name_in_message = result.final_output.is_name_in_message
  return GuardrailFunctionOutput(output_info={"found_name": result.final_output},tripwire_triggered=is_name_in_message)

In [35]:
careful_sale_manager = Agent(
  name="Careful Sales Manager",
  instructions=sales_manager_instructions,
  model="gpt-4o-mini",
  tools=tools,
  handoffs=[email_agent],
  input_guardrails=[guardrail_against_name]
)

message = "Send out a cold sales email addressed to Dear CEO from Alice" # this should trigger a tripwire because it has a name in it failing the guardrail check

with trace("Guardrail against names"):
    response = await Runner.run(careful_sale_manager, message)

# Lets Create structured output for our email agent to ensure that it outputs the subject and body in a specific format that we can then use to send the email.
- lets use email agent as an example to show how we can use structured outputs and guardrails to ensure that the email agent outputs the subject and body in a specific format that we can then use to send the email.

In [ ]:
class EmailOutput(BaseModel):
    subject: str
    body: str
    recipient: str
    sender: str

email_gaurdrail_agent= Agent(
  name="Email Manager Agent with Output",
  instructions=instructions,
  model="gpt-4o-mini",
  output_type=EmailOutput
)

In [37]:
# now lets create an output guardrail
from agents import output_guardrail


@output_guardrail
async def email_output_guardrail(ctx, agent, message):
  result = await Runner.run(email_gaurdrail_agent, message, context=ctx.context)
  email_output = result.final_output
  # we can add any logic here to check if the output is valid, for example we can check if the subject and body are not empty
  is_valid_email = bool(email_output.subject) and bool(email_output.body)
  return GuardrailFunctionOutput(output_info={"email_output": email_output}, tripwire_triggered=not is_valid_email)

In [39]:
# lets run the careful sales manager again but now with the output guardrail to ensure that the email agent outputs a valid email format
careful_sale_manager_with_output_guardrail = Agent(
  name="Careful Sales Manager with Output Guardrail",
  instructions=sales_manager_instructions,
  model="gpt-4o-mini",
  tools=tools,
  handoffs=[email_agent],
  input_guardrails=[guardrail_against_name],
  output_guardrails=[email_output_guardrail]
)

message_invalid_email = "Send out a cold sales email with no subject and no body to BOB" # this should trigger the output guardrail because the email output will not be valid
with trace("Guardrail against invalid email output"):
    response = await Runner.run(careful_sale_manager_with_output_guardrail, message_invalid_email)

InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire